<a href="https://colab.research.google.com/github/SLIIT-Y4-S2-DL-ORG/Deep-Learning-Assignment/blob/main/MLP/MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install tensorflow

In [4]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "Processed_Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_FILE = (
    PROJECT_ROOT
    / "Processed_Data"
    / "har_processed.npz"
)

print(DATA_FILE)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
data = np.load(DATA_FILE)

X_train = data["X_train"]
X_val = data["X_val"]
X_test = data["X_test"]

y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

activity_names = data[
    "activity_names"
]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

In [ ]:
X_train_mlp = X_train.reshape(
    X_train.shape[0],
    -1
)

X_val_mlp = X_val.reshape(
    X_val.shape[0],
    -1
)

X_test_mlp = X_test.reshape(
    X_test.shape[0],
    -1
)

print("MLP train:", X_train_mlp.shape)
print("MLP validation:", X_val_mlp.shape)
print("MLP test:", X_test_mlp.shape)

Dataset downloaded and extracted.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization
)

model = Sequential([
    Input(
        shape=(X_train_mlp.shape[1],)
    ),

    Dense(
        256,
        activation="relu"
    ),

    BatchNormalization(),

    Dropout(0.30),

    Dense(
        128,
        activation="relu"
    ),

    BatchNormalization(),

    Dropout(0.30),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        6,
        activation="softmax"
    )
])

model.summary()

 har.zip   sample_data	'UCI HAR Dataset.names'  'UCI HAR Dataset.zip'


In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

activity_labels.txt  features_info.txt	features.txt  README.txt  test	train
body_acc_x_train.txt
body_acc_y_train.txt
body_acc_z_train.txt
body_gyro_x_train.txt
body_gyro_y_train.txt
body_gyro_z_train.txt
total_acc_x_train.txt
total_acc_y_train.txt
total_acc_z_train.txt


In [ ]:
start_time = time.time()

history = model.fit(
    X_train_mlp,
    y_train,

    validation_data=(
        X_val_mlp,
        y_val
    ),

    epochs=100,
    batch_size=64,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)

training_time = (
    time.time() - start_time
)

print(
    f"Training time: "
    f"{training_time:.2f}s"
)

Training shape: (7352, 128, 9)
Test shape: (2947, 128, 9)


In [ ]:
test_loss, test_accuracy = (
    model.evaluate(
        X_test_mlp,
        y_test,
        verbose=0
    )
)

print(
    f"Test accuracy: "
    f"{test_accuracy:.4f}"
)

Original labels: [1 2 3 4 5 6]


In [ ]:
y_probability = model.predict(
    X_test_mlp
)

y_pred = np.argmax(
    y_probability,
    axis=1
)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=activity_names,
        digits=4
    )
)

[0 1 2 3 4 5]


In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=activity_names
)

fig, ax = plt.subplots(
    figsize=(10, 8)
)

disp.plot(
    ax=ax,
    xticks_rotation=45
)

plt.title(
    "MLP Confusion Matrix"
)

plt.show()

,Activity,Train,Test
0,WALKING,1226,496
1,WALKING_UPSTAIRS,1073,471
2,WALKING_DOWNSTAIRS,986,420
3,SITTING,1286,491
4,STANDING,1374,532
5,LAYING,1407,537
